# 01 · Preparação dos dados — São Paulo

**Pergunta do projeto:** crime e preço de imóvel têm correlação negativa? Ou seja, bairro mais violento, metro quadrado mais barato?

Este notebook só faz a ingestão. Ele existe separado porque é lento: os 5 arquivos de crime somam ~1 GB de `.xlsx` e uma única aba leva ~90 s para ser lida. O resultado é gravado em parquet, e os notebooks de análise leem o parquet em segundos.

### Fontes em `data/SP/`

| Fonte | Conteúdo | Granularidade |
|---|---|---|
| `Crimes/SPDadosCriminais_{2022..2026}.xlsx` | Boletins de ocorrência da SSP-SP | 1 linha por ocorrência, **estado inteiro** |
| `Imoveis/GUIAS*.xlsx` | Guias de ITBI pagas | 1 linha por transação, **só a capital** |

### O problema a resolver aqui

As duas bases não compartilham nenhuma chave geográfica. O ITBI tem o nome abreviado da rua e nada mais: seu campo `Bairro` só está preenchido em 61% das linhas e tem 5.116 valores distintos cheios de abreviação (`VL ANDRADE`, `JD LARANJAL`) e até nome de condomínio. Já os crimes têm bairro em 99,7% das linhas, mais latitude e longitude em 80%.

A saída é usar o dataset de crimes como **gazetteer** — um dicionário geográfico da cidade. Agrupando as ocorrências por nome de rua normalizado, cada rua ganha um bairro modal e uma coordenada mediana. Esse dicionário é então aplicado ao ITBI:

```
crime:  'AVENIDA BRIGADEIRO LUIS ANTONIO' + BELA VISTA + (-23.56, -46.64)
            ↓ normaliza: tira o tipo de via, expande BRIG → BRIGADEIRO
gazetteer['BRIGADEIRO LUIS ANTONIO'] = (BELA VISTA, -23.56, -46.64)
            ↓
ITBI:   'AV BRIG LUIS ANTONIO' → BELA VISTA, (-23.56, -46.64)
```

Casar bairro-com-bairro cobriria só 32% das transações. Via nome de rua a cobertura passa de 90% — e de brinde cada transação ganha uma coordenada aproximada, o que permite mapas de calor sem precisar de shapefile.

### Saídas

Três arquivos em `data/SP/cache/`: `crimes_sp.parquet`, `itbi_sp.parquet` e `gazetteer_ruas.parquet`. Os parquets guardam dados **já tipados e limpos** — os notebooks seguintes não repetem tratamento.

In [1]:
import gzip
import io
import re
import unicodedata
import urllib.request
from pathlib import Path

import geopandas as gpd
import pandas as pd
from openpyxl import load_workbook

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CRIMES = RAIZ / "data" / "SP" / "Crimes"
IMOVEIS = RAIZ / "data" / "SP" / "Imoveis"
CACHE = RAIZ / "data" / "SP" / "cache"
CACHE.mkdir(exist_ok=True)

sorted(p.name for p in CRIMES.glob("*.xlsx")) + sorted(p.name for p in IMOVEIS.glob("*.xlsx"))

['SPDadosCriminais_2022.xlsx',
 'SPDadosCriminais_2023.xlsx',
 'SPDadosCriminais_2024.xlsx',
 'SPDadosCriminais_2025.xlsx',
 'SPDadosCriminais_2026.xlsx',
 'GUIAS DE ITBI PAGAS (28012026) XLS.xlsx',
 'GUIAS DE ITBI PAGAS (30092026).xlsx',
 'GUIAS-DE-ITBI-PAGAS-2023.xlsx',
 'GUIAS-DE-ITBI-PAGAS-2024.xlsx',
 'GUIAS_DE_ITBI_PAGAS_12-2022.xlsx']

## Funções de apoio

`chave_rua` é o coração do gazetteer. Ela joga fora o tipo de via (`R`, `AV`, `TV`…), que é justamente o que difere entre as bases, e expande as abreviações de título (`BRIG` → `BRIGADEIRO`). O que sobra é o nome próprio da rua, que as duas bases escrevem igual.

`em_unicos` aplica uma função Python só nos valores distintos e devolve o resultado por *lookup*. São ~40 mil ruas distintas em ~2 milhões de linhas, então isso transforma 2 milhões de chamadas em 40 mil.

`cache` grava o resultado em parquet na primeira execução e lê de volta nas seguintes — é o que permite reabrir este notebook sem esperar 20 minutos.

In [2]:
TIPOS_VIA = {
    "R", "RUA", "AV", "AVENIDA", "TV", "TRAVESSA", "PC", "PRACA", "AL", "ALAMEDA",
    "EST", "ESTR", "ESTRADA", "LG", "LARGO", "VD", "VIADUTO", "VIA", "ROD", "RODOVIA",
    "MARGINAL", "BC", "BECO", "PSG", "PASSAGEM", "PASSEIO", "PTE", "PONTE", "ACS",
    "ACESSO", "COR", "CORREDOR", "ESC", "ESCADARIA", "TR", "TREVO", "BALAO", "COMP",
    "COMPLEXO", "PRQ", "PARQUE", "PQ", "JD", "JARDIM", "CJ", "CONJUNTO", "SIT", "MKD",
}

ABREVIACOES = {
    "S": "SAO", "STO": "SANTO", "STA": "SANTA", "D": "DOM", "PE": "PADRE",
    "PROF": "PROFESSOR", "PROFA": "PROFESSORA", "DR": "DOUTOR", "DRA": "DOUTORA",
    "CEL": "CORONEL", "GEN": "GENERAL", "CAP": "CAPITAO", "ENG": "ENGENHEIRO",
    "SEN": "SENADOR", "DEP": "DEPUTADO", "PRES": "PRESIDENTE", "CDE": "CONDE",
    "VISC": "VISCONDE", "BRIG": "BRIGADEIRO", "MAL": "MARECHAL", "REV": "REVERENDO",
    "MIN": "MINISTRO", "DES": "DESEMBARGADOR", "SGT": "SARGENTO", "TEN": "TENENTE",
    "MAJ": "MAJOR", "ALM": "ALMIRANTE", "N": "NOSSA", "SRA": "SENHORA",
    "MONS": "MONSENHOR", "VER": "VEREADOR", "COM": "COMENDADOR", "BR": "BARAO",
    "MQS": "MARQUES", "IRM": "IRMA", "CONS": "CONSELHEIRO", "PRINC": "PRINCESA",
}

VAZIOS = {"", "NULL", "NAN", "NONE"}


def sem_acento(texto):
    normalizado = unicodedata.normalize("NFKD", str(texto))
    return normalizado.encode("ascii", "ignore").decode().upper().strip()


def limpa_texto(valor):
    texto = sem_acento(valor)
    return None if texto in VAZIOS else texto


def chave_rua(nome):
    texto = limpa_texto(nome)
    if texto is None or "VEDACAO" in texto:
        return None
    palavras = re.sub(r"[^A-Z0-9 ]", " ", texto).split()
    while palavras and palavras[0] in TIPOS_VIA:
        palavras = palavras[1:]
    return " ".join(ABREVIACOES.get(p, p) for p in palavras) or None


def em_unicos(serie, funcao):
    return serie.map({valor: funcao(valor) for valor in serie.dropna().unique()})


def cache(nome, construir):
    destino = CACHE / f"{nome}.parquet"
    if destino.exists():
        print(f"{nome}: lendo do cache")
        return pd.read_parquet(destino)
    tabela = construir()
    tabela.to_parquet(destino, index=False)
    print(f"{nome}: {len(tabela):,} linhas gravadas em {destino.name}")
    return tabela


chave_rua("AV BRIG LUIS ANTONIO"), chave_rua("AVENIDA BRIGADEIRO LUÍS ANTÔNIO")

('BRIGADEIRO LUIS ANTONIO', 'BRIGADEIRO LUIS ANTONIO')

## 1 · Crimes

Três detalhes dos arquivos brutos obrigam a ler célula por célula em vez de usar `pd.read_excel` direto:

1. **A base é do estado inteiro.** Só ~40% das linhas são da capital, e é melhor descartar as outras durante a leitura do que carregar 600 mil linhas na memória para depois filtrar.
2. **O cabeçalho muda de ano para ano.** São três variações só nesses cinco arquivos: 2022 usa `DESCR_PERIODO`, os outros usam `DESC_PERIODO`; de 2022 a 2025 a grafia é `CIRCUNCRIÇÃO` (sem o "S" de *circunscrição*) e em 2026 virou `CIRCUNSCRIÇÃO`; 2022 ainda chama o município de `CIDADE`. Por isso cada coluna é localizada por **prefixo**, com grafias alternativas aceitas, e `acha_coluna` levanta erro nomeando o campo caso nenhuma case — melhor falhar explicando do que trazer a coluna errada.
3. **Os tipos vêm misturados.** `LATITUDE` traz float, o inteiro `0` e a string `'NULL'` na mesma coluna. A limpeza acontece antes de gravar o parquet, senão o próprio parquet não consegue decidir o tipo da coluna.

Usa-se a geografia *de circunscrição* (onde o fato ocorreu), não a da delegacia que registrou — uma ocorrência pode ser registrada na Delegacia Eletrônica e ter acontecido em qualquer lugar da cidade.

Coordenadas fora da janela da cidade (latitude -24,0 a -23,3; longitude -46,9 a -46,3) são erro de digitação e viram nulas.

In [3]:
CAMPOS_CRIME = {
    "municipio": ("NOME_MUNICIPIO_CIRCUN",),
    "data_ocorrencia": ("DATA_OCORRENCIA_BO",),
    "periodo": ("DESC_PERIODO", "DESCR_PERIODO"),
    "local": ("DESCR_SUBTIPOLOCAL",),
    "bairro": ("BAIRRO",),
    "logradouro": ("LOGRADOURO",),
    "lat": ("LATITUDE",),
    "lon": ("LONGITUDE",),
    "natureza": ("NATUREZA_APURADA",),
    "delegacia": ("NOME_DELEGACIA_CIRCUN",),
}

JANELA_CIDADE = {"lat": (-24.0, -23.3), "lon": (-46.9, -46.3)}


def acha_coluna(cabecalho, prefixos, aba):
    posicao = next((i for i, c in enumerate(cabecalho) if c.startswith(prefixos)), None)
    if posicao is None:
        raise KeyError(f"{aba}: nenhuma coluna começa com {prefixos}")
    return posicao


def ler_aba_crime(caminho, aba):
    planilha = load_workbook(caminho, read_only=True)
    linhas = planilha[aba].iter_rows(values_only=True)
    cabecalho = [sem_acento(c) for c in next(linhas)]
    posicao = {
        novo: acha_coluna(cabecalho, prefixos, aba)
        for novo, prefixos in CAMPOS_CRIME.items()
    }
    coluna_municipio = posicao["municipio"]
    dados = [
        [linha[i] for i in posicao.values()]
        for linha in linhas
        if linha[coluna_municipio] == "S.PAULO"
    ]
    planilha.close()
    return pd.DataFrame(dados, columns=list(posicao)).drop(columns="municipio")


def limpa_crimes(tabela):
    for coluna in ["bairro", "natureza", "periodo", "local", "delegacia"]:
        tabela[coluna] = em_unicos(tabela[coluna], limpa_texto).astype("category")

    tabela["logradouro"] = em_unicos(tabela["logradouro"], limpa_texto)
    tabela["rua_key"] = em_unicos(tabela["logradouro"], chave_rua)
    tabela["data_ocorrencia"] = pd.to_datetime(tabela["data_ocorrencia"], errors="coerce")
    tabela["ano"] = tabela["data_ocorrencia"].dt.year
    tabela["mes"] = tabela["data_ocorrencia"].dt.to_period("M").astype(str)

    for coluna, (minimo, maximo) in JANELA_CIDADE.items():
        valores = pd.to_numeric(tabela[coluna], errors="coerce")
        tabela[coluna] = valores.where(valores.between(minimo, maximo))

    return tabela[tabela["ano"].between(2022, 2026)].reset_index(drop=True)


def monta_crimes():
    partes = []
    for caminho in sorted(CRIMES.glob("SPDadosCriminais_*.xlsx")):
        planilha = load_workbook(caminho, read_only=True)
        abas = [a for a in planilha.sheetnames if re.fullmatch(r"(JAN-JUN|JUL-DEZ)_\d{4}", a)]
        planilha.close()
        for aba in abas:
            parte = ler_aba_crime(caminho, aba)
            print(f"  {aba}: {len(parte):,} ocorrências na capital")
            partes.append(parte)
    return limpa_crimes(pd.concat(partes, ignore_index=True))


crimes = cache("crimes_sp", monta_crimes)
crimes.shape

crimes_sp: lendo do cache


(2214258, 12)

In [4]:
crimes.notna().mean().mul(100).round(1).sort_values().to_frame("% preenchido")

,% preenchido
periodo,32.3
lat,81.0
lon,81.0
rua_key,91.9
local,99.5
bairro,99.7
logradouro,100.0
data_ocorrencia,100.0
natureza,100.0
delegacia,100.0


## 2 · ITBI

As guias de ITBI têm uma armadilha: **as abas mensais não têm cabeçalho no topo**. A linha com os nomes das colunas aparece lá na linha ~12.151, porque a planilha foi ordenada pela coluna A e o texto `Nº do Cadastro (SQL)` ordena depois de qualquer número. A leitura é então feita sem cabeçalho, com os 28 nomes declarados à mão (conferidos na aba `EXPLICAÇÕES` do próprio arquivo), e mantém só as linhas em que a primeira coluna é numérica — o que de quebra elimina as linhas vazias do rodapé.

Esse `SQL` é o número do imóvel no cadastro do IPTU e tem estrutura `setor (3) + quadra (3) + lote (4) + dígito (1)`. O **setor fiscal** (001 a 310) é uma divisão geográfica da cidade e está preenchido em 100% das linhas — fica guardado como geografia de reserva, caso o gazetteer falhe em alguma rua.

In [5]:
COLUNAS_ITBI = [
    "sql", "logradouro", "numero", "complemento", "bairro_itbi", "referencia", "cep",
    "natureza_transacao", "valor_transacao", "data_transacao", "vvr",
    "proporcao_transmitida", "vvr_proporcional", "base_calculo", "tipo_financiamento",
    "valor_financiado", "cartorio", "matricula", "situacao_sql", "area_terreno",
    "testada", "fracao_ideal", "area_construida", "uso_cod", "uso_desc", "padrao_cod",
    "padrao_desc", "acc",
]

NUMERICAS_ITBI = [
    "valor_transacao", "vvr", "vvr_proporcional", "base_calculo", "valor_financiado",
    "proporcao_transmitida", "area_terreno", "testada", "fracao_ideal",
    "area_construida", "acc", "uso_cod", "padrao_cod",
]


def limpa_itbi(tabela):
    tabela[NUMERICAS_ITBI] = tabela[NUMERICAS_ITBI].apply(pd.to_numeric, errors="coerce")
    tabela["sql"] = tabela["sql"].astype("int64").astype(str).str.zfill(11)
    tabela["setor_fiscal"] = tabela["sql"].str[:3]
    tabela["logradouro"] = em_unicos(tabela["logradouro"], limpa_texto)
    tabela["rua_key"] = em_unicos(tabela["logradouro"], chave_rua)
    tabela["bairro_itbi"] = em_unicos(tabela["bairro_itbi"], limpa_texto)
    tabela["data_transacao"] = pd.to_datetime(tabela["data_transacao"], errors="coerce")
    tabela["ano"] = tabela["data_transacao"].dt.year
    tabela["mes"] = tabela["data_transacao"].dt.to_period("M").astype(str)

    for coluna in ["natureza_transacao", "uso_desc", "padrao_desc", "setor_fiscal", "situacao_sql"]:
        tabela[coluna] = em_unicos(tabela[coluna], limpa_texto).astype("category")

    descartaveis = ["complemento", "referencia", "cartorio", "tipo_financiamento", "matricula"]
    return tabela.drop(columns=descartaveis).reset_index(drop=True)


def monta_itbi():
    partes = []
    for caminho in sorted(IMOVEIS.glob("*.xlsx")):
        planilha = load_workbook(caminho, read_only=True)
        abas = [a for a in planilha.sheetnames if re.fullmatch(r"[A-Z]{3}-\d{4}", a)]
        for aba in abas:
            linhas = planilha[aba].iter_rows(values_only=True)
            dados = [
                (list(linha) + [None] * 28)[:28]
                for linha in linhas
                if isinstance(linha[0], (int, float))
            ]
            partes.append(pd.DataFrame(dados, columns=COLUNAS_ITBI))
        planilha.close()
        print(f"  {caminho.name}: {len(abas)} meses")
    return limpa_itbi(pd.concat(partes, ignore_index=True))


itbi = cache("itbi_sp", monta_itbi)
itbi.shape

itbi_sp: lendo do cache


(943536, 27)

In [6]:
itbi.notna().mean().mul(100).round(1).sort_values().to_frame("% preenchido")

,% preenchido
bairro_itbi,61.1
acc,82.6
uso_desc,99.7
sql,100.0
cep,100.0
natureza_transacao,100.0
logradouro,100.0
numero,100.0
vvr,100.0
proporcao_transmitida,100.0


## 3 · Gazetteer de ruas

Para cada rua normalizada: o bairro mais frequente e a coordenada mediana das ocorrências registradas nela. A mediana é preferida à média porque é imune a uma coordenada errada isolada.

O bairro modal sai de um `groupby` de duas chaves ordenado por contagem — bem mais rápido que chamar `.mode()` dentro de um `agg`, que rodaria Python puro em cada um dos ~40 mil grupos.

In [7]:
def monta_gazetteer():
    base = crimes.dropna(subset=["rua_key", "bairro"])
    pares = base.groupby(["rua_key", "bairro"], observed=True).size().rename("n").reset_index()
    modal = (
        pares.sort_values("n")
        .groupby("rua_key", observed=True)
        .agg(bairro=("bairro", "last"), n_ocorrencias=("n", "sum"))
    )
    coordenadas = base.groupby("rua_key", observed=True)[["lat", "lon"]].median()
    return modal.join(coordenadas).reset_index()


gazetteer = cache("gazetteer_ruas", monta_gazetteer)
gazetteer.sort_values("n_ocorrencias", ascending=False).head(10)

gazetteer_ruas: lendo do cache


,rua_key,bairro,n_ocorrencias,lat,lon
37511,PAULISTA,BELA VISTA,21239,-23.562018,-46.655907
11989,CRUZEIRO DO SUL,SANTANA,15517,-23.516237,-46.625053
14025,DO ESTADO,BOM RETIRO,11606,-23.551932,-46.625802
5395,AUGUSTA,CONSOLACAO,11025,-23.553313,-46.655047
12347,DA LUZ,BOM RETIRO,10790,-23.534972,-46.635300
33091,MARIO DE ANDRADE,BARRA FUNDA,10562,-23.527406,-46.664291
43751,SENADOR TEOTONIO VILELA,CIDADE DUTRA,10468,-23.704761,-46.700849
12223,DA CONSOLACAO,CONSOLACAO,10222,-23.549329,-46.653231
12427,DA REPUBLICA,REPUBLICA,7975,-23.543408,-46.642790
43292,SAPOPEMBA,SAPOPEMBA,7652,-23.610180,-46.504422


## 4 · Cobertura do gazetteer

A métrica que decide se a estratégia funcionou: qual fração das transações de ITBI consegue herdar um bairro. Comparada com a alternativa óbvia — casar o campo `Bairro` do ITBI direto com o dos crimes.

In [8]:
cobertura = pd.DataFrame(
    {
        "estratégia": ["gazetteer de ruas", "campo bairro do ITBI", "setor fiscal"],
        "cobertura": [
            itbi["rua_key"].isin(set(gazetteer["rua_key"])).mean(),
            itbi["bairro_itbi"].isin(set(crimes["bairro"].dropna().unique())).mean(),
            itbi["setor_fiscal"].notna().mean(),
        ],
    }
)
cobertura.style.format({"cobertura": "{:.1%}"}).hide(axis="index")

estratégia,cobertura
gazetteer de ruas,92.7%
campo bairro do ITBI,31.5%
setor fiscal,100.0%


In [9]:
pd.DataFrame(
    [
        ("crimes_sp", len(crimes), crimes["mes"].min(), crimes["mes"].max()),
        ("itbi_sp", len(itbi), itbi["mes"].min(), itbi["mes"].max()),
        ("gazetteer_ruas", len(gazetteer), "—", "—"),
    ],
    columns=["tabela", "linhas", "início", "fim"],
)

,tabela,linhas,início,fim
0,crimes_sp,2214258,2022-01,2026-08
1,itbi_sp,943536,1994-06,2026-09
2,gazetteer_ruas,48214,—,—


## 5 · Geometria da cidade (GeoSampa)

O gazetteer da seção 3 resolve o problema de localizar transações, mas por **inferência** — nome de rua e CEP. O GeoSampa, geoportal oficial da Prefeitura de São Paulo, oferece a geometria da cidade via WFS, e com ela a inferência deixa de ser necessária no lado do imóvel.

A camada decisiva é **`setor_fiscal`**: os 309 setores fiscais do cadastro do IPTU. Os três primeiros dígitos do `SQL` de cada guia de ITBI *são* esse código, o que significa uma chave de junção **exata e completa** — medido: 100,00% das transações casam com um polígono. Nenhum nome de rua, nenhum CEP, nenhuma normalização de texto.

Três camadas são baixadas:

| camada | unidades | uso |
|---|---|---|
| `distrito_municipal` | 96 | unidade nomeada para mapas e para a checagem do notebook 05 |
| `setor_fiscal` | 309 | chave exata para o ITBI, via `SQL` |
| — | — | tabela derivada `setor → distrito`, por maior sobreposição de área |

**Dois detalhes técnicos que o código trata:**

1. **`srsName=EPSG:4326` na requisição.** O GeoSampa publica em EPSG:31983 (UTM 23S, em metros). Pedir a reprojeção ao servidor evita fazê-la aqui e garante lat/long direto.
2. **Simplificação a 20 metros.** A geometria bruta dos distritos tem 4 MB de GeoJSON, o que pesaria num gráfico interativo embutido no notebook. Simplificar com tolerância de 20 m derruba para 247 KB — 94% menor — e na escala da cidade é visualmente indistinguível. A simplificação é feita em EPSG:31983, porque tolerância em metros só faz sentido num CRS métrico; o cálculo de área, pelo mesmo motivo, também.

O `setor → distrito` é resolvido por **maior sobreposição de área**: o setor fiscal não respeita a divisa de distrito, então cada setor é atribuído ao distrito com que mais se sobrepõe. É determinístico e dispensa qualquer escolha manual. Checagem de sanidade esperada: setor 001 → Sé, setores 002 e 003 → Brás, setor 006 → República.

In [10]:
WFS_GEOSAMPA = "https://wms.geosampa.prefeitura.sp.gov.br/geoserver/geoportal/wfs"
CRS_METRICO = 31983
TOLERANCIA_M = 20

CAMADAS = {
    "distritos_sp": ("distrito_municipal", ["cd_distrito_municipal", "nm_distrito_municipal"]),
    "setores_fiscais": ("setor_fiscal", ["cd_setor_fiscal"]),
}


def baixa_camada(camada, colunas):
    consulta = (
        f"{WFS_GEOSAMPA}?service=WFS&version=1.0.0&request=GetFeature"
        f"&typeName={camada}&outputFormat=application/json&srsName=EPSG:4326"
    )
    pedido = urllib.request.Request(
        consulta, headers={"Accept-Encoding": "identity", "User-Agent": "Mozilla/5.0"}
    )
    with urllib.request.urlopen(pedido, timeout=300) as resposta:
        conteudo = resposta.read()
    if conteudo[:2] == b"\x1f\x8b":
        conteudo = gzip.decompress(conteudo)
    geometrias = gpd.read_file(io.BytesIO(conteudo))[colunas + ["geometry"]]
    geometrias["geometry"] = (
        geometrias.to_crs(CRS_METRICO).simplify(TOLERANCIA_M).to_crs(4326)
    )
    return geometrias


def cache_geo(nome):
    destino = CACHE / f"{nome}.geojson"
    if destino.exists():
        print(f"{nome}: lendo de {destino.name}")
        return gpd.read_file(destino)
    camada, colunas = CAMADAS[nome]
    geometrias = baixa_camada(camada, colunas)
    geometrias.to_file(destino, driver="GeoJSON")
    print(f"{nome}: {len(geometrias)} feições baixadas do GeoSampa ({destino.stat().st_size / 1024:,.0f} KB)")
    return geometrias


distritos = cache_geo("distritos_sp")
setores = cache_geo("setores_fiscais")
setores["setor_fiscal"] = setores["cd_setor_fiscal"].astype(str).str.zfill(3)
setores = setores.dissolve(by="setor_fiscal").reset_index()[["setor_fiscal", "geometry"]]

print(f"\n{len(distritos)} distritos · {len(setores)} setores fiscais")
print(f"ITBI com setor fiscal presente na malha: {itbi['setor_fiscal'].astype(str).isin(set(setores['setor_fiscal'])).mean():.2%}")

distritos_sp: 96 feições baixadas do GeoSampa (221 KB)


setores_fiscais: 312 feições baixadas do GeoSampa (544 KB)

96 distritos · 309 setores fiscais
ITBI com setor fiscal presente na malha: 100.00%


In [11]:
def monta_setor_para_distrito():
    cruzamento = gpd.overlay(setores, distritos, how="intersection", keep_geom_type=True)
    cruzamento["area_m2"] = cruzamento.to_crs(CRS_METRICO).area
    return (
        cruzamento.sort_values("area_m2")
        .groupby("setor_fiscal")
        .agg(distrito=("nm_distrito_municipal", "last"), area_m2=("area_m2", "last"))
        .reset_index()
    )


setor_para_distrito = cache("setor_para_distrito", monta_setor_para_distrito)

print(f"\ndistritos alcançados: {setor_para_distrito['distrito'].nunique()} de {len(distritos)}")
print("\nchecagem de sanidade — os setores do centro histórico:")
print(setor_para_distrito[setor_para_distrito["setor_fiscal"].isin(["001", "002", "003", "004", "006"])]
      [["setor_fiscal", "distrito"]].to_string(index=False))

setor_para_distrito: 309 linhas gravadas em setor_para_distrito.parquet

distritos alcançados: 94 de 96

checagem de sanidade — os setores do centro histórico:
setor_fiscal  distrito
         001        SE
         002      BRAS
         003      BRAS
         004   CAMBUCI
         006 REPUBLICA


### Ressalva a carregar para a análise

Uma rua só entra no gazetteer se alguma ocorrência foi registrada nela. Isso significa que as transações **não casadas** tendem a estar em ruas pequenas ou tranquilas, o que enviesa levemente a amostra analisada na direção de ruas mais movimentadas. No nível de bairro o efeito é pequeno — todo bairro tem ocorrências —, mas a ressalva fica registrada e é retomada no notebook de EDA.

➡️ Próximo: `02_eda.ipynb`.